**01 · Geometrías, escenarios y perturbaciones**

Ejecutar desde el repositorio point-field con `python -m pip install -e ".[learning,bench,dev]"` y Jupyter. Las salidas nuevas se guardan en `runs/tesis_figuras/`. No reemplazar resultados históricos sin comprobar el protocolo.

In [ ]:
from pathlib import Path
import sys, subprocess, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'benchmarks'/'common.py').exists() and (p/'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError("Abre Jupyter desde la raíz de point-field o una subcarpeta del repositorio.")
sys.path.insert(0, str(ROOT/'benchmarks'))
OUT = ROOT/'runs'/'tesis_figuras'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'figure.dpi': 120, 'savefig.dpi': 300, 'pdf.fonttype': 42})
COLORS = ['#21618c','#c65036','#63933e','#8b65a5','#c49a38','#458e91']
def save(fig, name):
    for ext in ('pdf','png'):
        fig.savefig(OUT/f'{name}.{ext}', bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)
def points(ax, X, y=None):
    if y is None: ax.scatter(*X.T, s=5, c=COLORS[0], linewidths=0)
    else:
        for j,c in enumerate(np.unique(y)):
            ax.scatter(*X[y==c].T, s=5, c='#b9b9b9' if c==-1 else COLORS[j%len(COLORS)], linewidths=0)
    ax.set_xticks([]); ax.set_yticks([])
    if X.shape[1]==2: ax.set_aspect('equal', adjustable='box')
    else: ax.set_zticks([]); ax.set_box_aspect((1,1,1)); ax.view_init(elev=23, azim=35)
def run_script(name, *args):
    subprocess.run([sys.executable, str(ROOT/'benchmarks'/name), *map(str,args)], cwd=ROOT, check=True)
print('Repositorio:', ROOT, '\nSalidas:', OUT)
try:
    commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
except (OSError, subprocess.CalledProcessError):
    commit='No disponible'
print('Commit:',commit)

**1. Primitivas 2D.** Distinguir curvas de regiones rellenas. Las clases geométricas se muestrean con la misma semilla; el catálogo no constituye un benchmark de clustering.

In [ ]:
from pointfield.geometry.shapes2d import Circle, Disk, Annulus, Ellipse, Rectangle, Triangle, Star, Polygon, Line, Crescent, Spiral
geometries=[('Circunferencia',Circle()),('Disco',Disk()),('Anillo',Annulus(.5,1)),('Elipse',Ellipse(1,.5)),('Borde elíptico',Ellipse(1,.5,filled=False)),('Rectángulo',Rectangle(2,1)),('Triángulo',Triangle()),('Estrella',Star()),('Polígono no convexo',Polygon([[0,0],[2,0],[2,2],[1,.8],[0,2]])),('Línea',Line()),('Luna',Crescent()),('Espiral',Spiral(a=.3,b=.25,turns=2,kind='logarithmic'))]
fig,axes=plt.subplots(2,6,figsize=(12,4.3),constrained_layout=True)
for ax,(label,g) in zip(axes.flat,geometries):
    X=g.sample(500,seed=0); points(ax,X); ax.set_title(f'{label}\ndim. intrínseca {g.intrinsic_dimension}')
save(fig,'01_primitivas_2d')

**2. Primitivas 3D.** Mostrar superficies y curvas espaciales por separado de los datos 2D.

In [ ]:
from pointfield.geometry.shapes3d import Sphere,Cube,Torus,Cylinder,Cone,Ellipsoid,Helix,Pyramid
geometries3=[('Esfera',Sphere()),('Cubo',Cube()),('Toro',Torus()),('Cilindro',Cylinder()),('Cono',Cone()),('Elipsoide',Ellipsoid()),('Hélice',Helix(turns=3)),('Pirámide',Pyramid())]
fig=plt.figure(figsize=(12,6),constrained_layout=True)
for j,(label,g) in enumerate(geometries3):
    ax=fig.add_subplot(2,4,j+1,projection='3d'); points(ax,g.sample(700,seed=0)); ax.set_title(label)
save(fig,'01_primitivas_3d')

**3. Escenarios base y reservados.** Los colores son etiquetas de referencia, no predicciones.

In [ ]:
from common import MAIN_CASES,HELD_OUT_CASES,load_case
cases=MAIN_CASES+HELD_OUT_CASES
fig,axs=plt.subplots(2,3,figsize=(9,6),constrained_layout=True)
for ax,c in zip(axs.flat,cases):
    ds=load_case(c,'base',100); points(ax,ds.X,ds.y); ax.set_title(c.replace('_',' ') + (' (reservado)' if c in HELD_OUT_CASES else ''))
save(fig,'01_escenarios')

**4. Perturbaciones.** Mantener el mismo sistema de coordenadas en los cuatro paneles; la referencia solo identifica el grupo.

In [ ]:
from pointfield.datasets import make
from pointfield.perturbations import add_noise,add_occlusion,add_variable_density
base=make('no_convexo',seed=0)
variants=[('Original',base),('Ruido: 0.06 RMS',add_noise(base,.06,seed=1)),('Oclusión: 25%',add_occlusion(base,.25,seed=1)),('Densidad: 1 a 0.15',add_variable_density(base,.15,seed=1))]
fig,axs=plt.subplots(1,4,figsize=(12,3.3),constrained_layout=True)
xy=np.vstack([ds.X for _,ds in variants]); lo=xy.min(0)-.1; hi=xy.max(0)+.1
for ax,(label,ds) in zip(axs,variants):
    points(ax,ds.X,ds.y);ax.set(xlim=(lo[0],hi[0]),ylim=(lo[1],hi[1]),title=f'{label}\nN={len(ds.X)}')
save(fig,'01_perturbaciones')